In [1]:
from core.models.model_module_infer import model_module
inference_tester = model_module(
    data_dir='checkpoints/',
    pth=['CoDi_encoders.pth'],
    fp16=False
).cuda().eval()

net = inference_tester.net
print("✅ CoDi 编码器加载完成")


ModuleNotFoundError: No module named 'core'

In [ ]:
import torch
import torchaudio
import torchvision.transforms as T
from PIL import Image
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = inference_tester.net

def _l2n(x): return torch.nn.functional.normalize(x, dim=-1)


In [ ]:
# 文本
def encode_text(list_of_str):
    with torch.no_grad():
        z = net.clip_encode_text(list_of_str).to(device)
        return _l2n(z)

# 图像
def encode_image(paths):
    outs = []
    tfm = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
    for p in paths:
        img = Image.open(p).convert('RGB')
        x = tfm(img).unsqueeze(0).to(device) * 2 - 1  # 归一化到 [-1,1]
        with torch.no_grad():
            f = net.clip_encode_vision(x)
            outs.append(_l2n(f))
    return torch.cat(outs, 0)

# 音频
def encode_audio(paths, target_sr=16000, seconds=10.0):
    outs = []
    T_len = int(target_sr * seconds)
    for p in paths:
        wav, sr = torchaudio.load(p)
        wav = wav.mean(0, keepdim=True)
        if sr != target_sr:
            wav = torchaudio.functional.resample(wav, sr, target_sr)
        if wav.size(-1) >= T_len:
            wav = wav[..., :T_len]
        else:
            wav = torch.nn.functional.pad(wav, (0, T_len - wav.size(-1)))
        wav = wav.unsqueeze(0).to(device)
        with torch.no_grad():
            f = net.clap_encode_audio(wav)
            outs.append(_l2n(f))
    return torch.cat(outs, 0)


In [ ]:
# ========= 随机图片 + 随机音频 下载/生成 并保存到本地 =========
import os, random
from pathlib import Path

import torch
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from PIL import Image

import torchaudio

random.seed(0)

IMG_DIR = Path("rand_data/images")
AUD_DIR = Path("rand_data/audios")
IMG_DIR.mkdir(parents=True, exist_ok=True)
AUD_DIR.mkdir(parents=True, exist_ok=True)

print("保存目录：")
print(" -", IMG_DIR.resolve())
print(" -", AUD_DIR.resolve())

# ---------------------------
# 1) 随机图片（CIFAR10）
# ---------------------------
print("\n[1/2] 下载 CIFAR-10 随机图片 ...")
try:
    cifar = CIFAR10(root="data", train=False, download=True)
    idxs = list(range(len(cifar)))
    random.shuffle(idxs)

    N_IMG = 16  # 保存多少张
    resample_bicubic = getattr(Image, "Resampling", Image).BICUBIC
    upscale = T.Resize(256, interpolation=resample_bicubic)

    saved_imgs = []
    for k in range(N_IMG):
        img, label = cifar[idxs[k]]   # PIL.Image, label 0..9
        img_big = upscale(img)        # 放大到 256x256 便于查看
        outp = IMG_DIR / f"img_{k:02d}_cls{label}.jpg"
        img_big.save(outp, quality=95)
        saved_imgs.append(str(outp))

    print(f"✅ 已保存 {len(saved_imgs)} 张图片到 {IMG_DIR}")
except Exception as e:
    print("⚠️ CIFAR-10 下载失败，改为本地生成随机彩色方块：", e)
    saved_imgs = []
    N_IMG = 16
    for k in range(N_IMG):
        img = Image.new("RGB", (256, 256), (random.randint(0,255), random.randint(0,255), random.randint(0,255)))
        outp = IMG_DIR / f"img_{k:02d}_synthetic.jpg"
        img.save(outp, quality=95)
        saved_imgs.append(str(outp))
    print(f"✅ 已生成 {len(saved_imgs)} 张合成图片到 {IMG_DIR}")

# ---------------------------
# 2) 随机音频（YESNO 优先；若无则合成哔声）
# ---------------------------
print("\n[2/2] 下载 YESNO 随机音频（若不可用则本地合成） ...")
saved_auds = []
TARGET_SR = 16000

def synth_beep_wav(path, sr=TARGET_SR, seconds=1.0, freq=440.0):
    import math
    import numpy as np
    t = np.linspace(0, seconds, int(sr*seconds), endpoint=False)
    wav = 0.2*np.sin(2*math.pi*freq*t).astype("float32")
    ten = torch.from_numpy(wav).unsqueeze(0)  # [1, T]
    torchaudio.save(str(path), ten, sr)

try:
    # 大多数 torchaudio 版本都带 YESNO；若失败会进入 except 分支
    from torchaudio.datasets import YESNO
    ds = YESNO(root="data", download=True)
    idxs = list(range(len(ds)))
    random.shuffle(idxs)

    N_AUD = min(10, len(ds))
    for k in range(N_AUD):
        wav, sr, labels = ds[idxs[k]]
        # 转 16k 单声道
        if sr != TARGET_SR:
            wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
        if wav.dim() == 2 and wav.size(0) > 1:
            wav = wav.mean(0, keepdim=True)
        outp = AUD_DIR / f"aud_{k:02d}_labels{''.join(map(str, labels))}.wav"
        torchaudio.save(str(outp), wav, TARGET_SR)
        saved_auds.append(str(outp))
    print(f"✅ 已保存 {len(saved_auds)} 条音频到 {AUD_DIR}")
except Exception as e:
    print("⚠️ torchaudio YESNO 不可用，改为合成哔声：", e)
    freqs = [330, 440, 550, 660, 770, 880]
    for k, f in enumerate(freqs):
        outp = AUD_DIR / f"aud_{k:02d}_beep{f}.wav"
        synth_beep_wav(outp, sr=TARGET_SR, seconds=1.2, freq=float(f))
        saved_auds.append(str(outp))
    print(f"✅ 已合成 {len(saved_auds)} 条哔声到 {AUD_DIR}")

print("\n完成！示例文件：")
print(" - image:", saved_imgs[0] if saved_imgs else "无")
print(" - audio:", saved_auds[0] if saved_auds else "无")
